# 65. 图表结构与Hover

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 2 / 18 步：建立交互图结构与 Hover 体验**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** Plotly 模块入门  →  **本章任务：** 图表结构与Hover  →  **下一步：** 交互折线图（px.line）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

看静态报表时，图表往往是一张定格的图片，想看清某根折线背后的具体数值，还得回到数据表里一行行翻。



## 本章目标

学完本章，你将能够：

- **理解**：理解「图表结构与Hover」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「图表结构与Hover」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「图表结构与Hover」并读出其中的结论。


## 65.1 适用场景

**背景引入**：看静态报表时，图表往往是一张定格的图片，想看清某根折线背后的具体数值，还得回到数据表里一行行翻。Plotly 画出来的图天生带着缩放、悬浮、图例筛选这些交互能力——鼠标一移上去，数值、名称和单位立刻显现，双击还能聚焦到数据点。对销售漏斗、月度趋势这类信息密集的分析，交互图表让读者自己动手探索，而不是被动接受一张模板化的图。（打个比方：Figure 就像一栋楼，每个 trace 是一间房（装一组数据），layout 是楼里的装修和指示牌——标题、坐标轴、图例。加一间房不砸另一间房，改装修不动数据，所以才能一个个往上加 trace、再统一设置布局。）

需要缩放、悬浮、图例筛选或导出独立HTML的交互图表。


## 65.2 数据结构

优先使用长表；Plotly Express快速建图，Graph Objects精细控制。


## 65.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 hovermode="x unified" 改为 hovermode="closest"，观察悬浮信息聚合方式的变化
2. 修改 template="plotly_white" 为 "plotly_dark"，对比不同模板的视觉风格
3. 修改 hovertemplate 自定义悬停信息格式，说明交互提示对精确读值的作用


## 65.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.line()`、`fig.update_layout()`、`fig.show()`、`data[0]` | 需要缩放、悬浮、图例筛选或导出独立HTML的交互图表。 | 把Hover当作唯一标签 |
| 进阶变体 | `go.Figure()`、`fig.add_trace()`、`go.Scatter()`、`fig.update_layout()` | 在基础图表上增加分组、注释、布局或交互 | 图表初始视图缺少结论 |
| 关键参数 | `data_frame` | 数据 | 把Hover当作唯一标签 |
| 关键参数 | `x/y` | 位置 | 图表初始视图缺少结论 |
| 关键参数 | `color` | 分组 | 在大数据上一次渲染过多点 |
| 关键参数 | `hover_data` | 悬浮信息 | 把Hover当作唯一标签 |


## 65.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Flights {
            len(flights):,        } | Gapminder {
                len(gapminder):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 65.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.line(
    monthly, x="month", y="sales", markers=True, title="Plotly Figure基础"
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元)", hovermode="x unified"
)
fig.show()

print("Trace数量:", len(fig.data))
print("第一个Trace类型:", fig.data[0].type)


**练一练**：把上面「基础图表」的折线图复制到下面，将 `hovermode` 从 `"x unified"` 改成 `"closest"`，感受悬浮提示从「聚合整列」变成「只看最近一个点」的区别。先运行脚手架自检（此刻会报 NameError 属正常），再在注释处补全并运行，观察图表与自检结果如何变化。


In [ ]:
# 请在下方填写代码：用 monthly 画一条折线图，并把 hovermode 改成 closest
# 参考上一格 cell 7 的写法：px.line(monthly, x="month", y="sales", markers=True)
# ---- 请在下方填写代码 ----
# fig = px.line(monthly, x="month", y="sales", markers=True, ...)  # 取消注释并补全


In [ ]:
# 完整答案：只改 hovermode，观察悬浮从聚合到单点显示的变化
fig = px.line(
    monthly,
    x="month",
    y="sales",
    markers=True,
    title="Plotly Figure基础（练一练）",
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元）", hovermode="closest"
)


## 65.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["sales"],
        mode="lines+markers",
        name="销售额",
    )
)
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["profit"],
        mode="lines+markers",
        name="利润",
    )
)
fig.update_layout(
    title="Graph Objects多Trace示例",
    xaxis_title="月份",
    yaxis_title="金额（万元）",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()


## 65.8 参数说明

- data_frame：数据
- x/y：位置
- color：分组
- hover_data：悬浮信息


## 65.9 结果解读

Hover补充精确值，缩放帮助局部检查，图例可显示或隐藏序列。


## 65.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 65.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 65.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 65.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 65.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 65.12 易错点提醒

- 把Hover当作唯一标签
- 图表初始视图缺少结论
- 在大数据上一次渲染过多点


## 65.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 65.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：给 hover 增加一个额外字段，让悬停信息更丰富
# 【目标】hover 是 Plotly 交互的核心，练习往悬停提示里加更多信息。
import plotly.express as px

# 起点示例(已可运行)：用 hover_data 控制悬停里显示哪些字段。
fig = px.line(
    monthly,
    x="month",
    y="sales",
    markers=True,
    hover_data={"month": False, "sales": ":.1f"},
    title="Plotly Figure基础（含悬停）",
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元）", hovermode="x unified"
)
fig.show()

# ---- 反思记录：加上悬停信息，读图时多获得了什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
fig = px.bar(
    regional,
    x="region",
    y="sales",
    color="channel",
    barmode="group",
    title="区域渠道销售",
)
fig.update_traces(
    hovertemplate="%{x}<br>销售额 %{y} 万元<extra>%{fullData.name}</extra>"
)
fig.update_layout(xaxis_title="区域", yaxis_title="销售额（万元）")
fig.show()


## 65.15 小结

理解Plotly Figure、Trace、Layout、交互模式和Hover信息。


### 65.15.1 你已经掌握

- 判断图表结构与Hover的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 65.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `data_frame` | 数据 |
| `x/y` | 位置 |
| `color` | 分组 |
| `hover_data` | 悬浮信息 |


### 65.15.3 需要注意

- 把Hover当作唯一标签
- 图表初始视图缺少结论
- 在大数据上一次渲染过多点


### 65.15.4 完成检查

- [ ] 能判断什么问题适合使用图表结构与Hover
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 65.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
